# BirdCLEF 2026 — Label-Head Ensemble

**Strategy:** Blend Perch zero-shot predictions with a fine-tuned label-head.

- Head input: Perch's 234-dim species logits (indexed to BirdCLEF 2026 targets)
- Head: small MLP 234→256→234 trained as calibration layer
- Ensemble: `alpha × perch_zero_shot + (1-alpha) × label_head_calibrated`

Upload `checkpoints/perch-label-head/best_head.weights.h5` as Kaggle dataset `birdclef2026-label-head`.
Also upload `checkpoints/label-head-pseudo/best_head.weights.h5` if available.

In [ ]:
import subprocess, sys

def _tf_version():
    try:
        import tensorflow as tf
        return tuple(int(x) for x in tf.__version__.split('.')[:2])
    except Exception:
        return (0, 0)

if _tf_version() < (2, 20):
    import os
    tb_wheel = '/kaggle/input/bc26-tensorflow-2-20-0/wheel/tensorboard-2.20.0-py3-none-any.whl'
    tf_wheel = '/kaggle/input/bc26-tensorflow-2-20-0/wheel/tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
    if os.path.isfile(tb_wheel):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', tb_wheel], check=False)
    if os.path.isfile(tf_wheel):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', tf_wheel], check=False)
    else:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tensorflow==2.20.0'], check=False)

import tensorflow as tf
assert tuple(int(x) for x in tf.__version__.split('.')[:2]) >= (2, 20), f'TF 2.20+ required, got {tf.__version__}'
print(f'TensorFlow {tf.__version__}')
tf.experimental.numpy.experimental_enable_numpy_behavior()

In [ ]:
import glob, os, re, time
import h5py
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf

START          = time.time()
TERMINATE_TIME = START + 8.5 * 3600

## Config

In [ ]:
DATA_DIR   = '/kaggle/input/birdclef-2026'
PERCH_DIR  = '/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1'

# Label-head weights  (234→256→234 head trained on Perch label features)
WEIGHTS_PATH = '/kaggle/input/birdclef2026-label-head/best_head.weights.h5'

# Ensemble blend weight for Perch zero-shot (0.0 = pure label-head, 1.0 = pure Perch zero-shot)
ALPHA = 0.4    # Perch zero-shot weight

NUM_CLASSES   = 234
HIDDEN_DIM    = 256    # label-head uses 256-dim hidden (input is 234-dim)
SAMPLE_RATE   = 32_000
CLIP_DURATION = 5
CLIP_SAMPLES  = SAMPLE_RATE * CLIP_DURATION
BATCH_SIZE    = 64

TEST_DIR  = os.path.join(DATA_DIR, 'test_soundscapes')
TRAIN_SC  = os.path.join(DATA_DIR, 'train_soundscapes')
SC_DIR    = TEST_DIR if glob.glob(os.path.join(TEST_DIR, '*.ogg')) else TRAIN_SC
print(f'Soundscapes dir: {SC_DIR}')
print(f'Files: {len(glob.glob(os.path.join(SC_DIR, "*.ogg")))}')

## Perch Species Mapping

In [ ]:
bc_labels = pd.read_csv(os.path.join(PERCH_DIR, 'assets/labels.csv'))
bc_labels = (bc_labels.reset_index()
             .rename({'inat2024_fsd50k': 'scientific_name', 'index': 'bc_index'}, axis=1)
             .set_index('scientific_name'))
N_PERCH = len(bc_labels)

taxonomy       = pd.read_csv(os.path.join(DATA_DIR, 'taxonomy.csv'))
sample_sub     = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
primary_labels = sample_sub.columns[1:].tolist()

mapping = taxonomy.join(bc_labels, on='scientific_name', how='left')
mapping.bc_index = mapping.bc_index.fillna(N_PERCH).astype(int)
mapping = mapping[['primary_label', 'bc_index']].set_index('primary_label')

# Indices into Perch label output; OOV → N_PERCH (will be 0 after pad)
perch_indices = [int(mapping.loc[pl][0]) if pl in mapping.index else N_PERCH
                 for pl in primary_labels]
n_covered = sum(1 for i in perch_indices if i < N_PERCH)
print(f'Perch covers {n_covered} / {NUM_CLASSES} target species')

## Load Perch + Label-Head

In [ ]:
print('Loading Perch …')
perch = tf.saved_model.load(PERCH_DIR)
sig   = perch.signatures['serving_default']

# Classification head: 234 → 256 → 234 (label-head architecture)
class ClassificationHead(tf.keras.Model):
    def __init__(self, num_classes, hidden_dim=256):
        super().__init__()
        self.fc1  = tf.keras.layers.Dense(hidden_dim)
        self.act  = tf.keras.layers.Activation('relu')
        self.drop = tf.keras.layers.Dropout(0.0)   # no dropout at inference
        self.fc2  = tf.keras.layers.Dense(num_classes)
    def call(self, x, training=False):
        x = self.fc1(x); x = self.act(x); x = self.drop(x, training=training)
        return self.fc2(x)

head = ClassificationHead(NUM_CLASSES, HIDDEN_DIM)
# Build with 234-dim input (Perch label features)
_ = head(tf.zeros((1, NUM_CLASSES)), training=False)

with h5py.File(WEIGHTS_PATH, 'r') as wf:
    head.fc1.kernel.assign(wf['fc1']['vars']['0'][:])
    head.fc1.bias.assign(  wf['fc1']['vars']['1'][:])
    head.fc2.kernel.assign(wf['fc2']['vars']['0'][:])
    head.fc2.bias.assign(  wf['fc2']['vars']['1'][:])
print(f'Label-head weights loaded ← {WEIGHTS_PATH}')

## Inference

In [ ]:
@tf.function(input_signature=[tf.TensorSpec([None, CLIP_SAMPLES], tf.float32)])
def predict_label_ensemble(clips):
    """Returns blended prediction: alpha * perch_zero_shot + (1-alpha) * label_head_calibrated."""
    out = sig(inputs=clips)
    # Pad so OOV index → 0
    label = tf.pad(out['label'], [[0, 0], [0, 1]])          # (N, N_PERCH+1)
    features = tf.gather(label, perch_indices, axis=1)       # (N, 234) raw logits
    
    # Perch zero-shot: sigmoid of raw label logits
    perch_probs = tf.sigmoid(features)
    
    # Label-head calibrated: pass features through trained head
    head_logits = head(tf.stop_gradient(features), training=False)
    head_probs  = tf.sigmoid(head_logits)
    
    # Weighted blend
    return ALPHA * perch_probs + (1.0 - ALPHA) * head_probs


def process_soundscape(filepath):
    ss_id  = re.sub(r'\.ogg$', '', os.path.basename(filepath), flags=re.IGNORECASE)
    audio, _ = librosa.load(filepath, sr=SAMPLE_RATE, mono=True)
    audio  = audio.astype(np.float32)
    n_segs = len(audio) // CLIP_SAMPLES
    if n_segs == 0:
        return [], np.empty((0, NUM_CLASSES))

    clips   = np.stack([audio[i*CLIP_SAMPLES:(i+1)*CLIP_SAMPLES] for i in range(n_segs)])
    row_ids = [f'{ss_id}_{(i+1)*CLIP_DURATION}' for i in range(n_segs)]

    all_preds = []
    for i in range(0, len(clips), BATCH_SIZE):
        batch  = tf.constant(clips[i:i+BATCH_SIZE], dtype=tf.float32)
        preds  = predict_label_ensemble(batch)
        all_preds.append(preds.numpy())

    return row_ids, np.concatenate(all_preds, axis=0)


ogg_files = sorted(glob.glob(os.path.join(SC_DIR, '*.ogg')))
all_row_ids, all_preds = [], []

for k, fpath in enumerate(ogg_files):
    if time.time() > TERMINATE_TIME:
        print(f'Time limit reached at {k}/{len(ogg_files)}')
        break
    if k % 200 == 0:
        print(f'[{k+1}/{len(ogg_files)}] elapsed={(time.time()-START)/60:.1f}m')
    try:
        row_ids, preds = process_soundscape(fpath)
    except Exception as e:
        print(f'ERROR {os.path.basename(fpath)}: {e}'); continue
    if row_ids:
        all_row_ids.extend(row_ids)
        all_preds.append(preds)

print(f'Done: {len(all_row_ids)} rows in {(time.time()-START)/60:.1f}m')

## Build Submission

In [ ]:
predictions = np.concatenate(all_preds, axis=0)
submission  = pd.DataFrame(predictions, columns=primary_labels)
submission.insert(0, 'row_id', all_row_ids)
submission  = sample_sub[['row_id']].merge(submission, on='row_id', how='left').fillna(0.0)
submission.to_csv('submission.csv', index=False)
print(f'submission.csv: {submission.shape[0]} rows × {len(primary_labels)} species')
print(f'Elapsed: {(time.time()-START)/60:.1f} min')
submission.head(5)